# Week 10 - LangGraph 상태(State) 관리 심화

## Part 1. State 확장과 리듀서

In [1]:
from typing import Annotated
from operator import add
from langgraph.graph import add_messages
from langgraph.graph import MessagesState

class AgentState(MessagesState):
    # 리듀서 없이: 마지막 값으로 덮어씀
    query_type: str
    # add_messages: 대화를 누적
    messages: Annotated[list, add_messages]
    # add: 검색 결과를 리스트에 누적
    sources: Annotated[list, add]

## Part 2. State에 무엇을 담을까

In [2]:
from langgraph.graph import MessagesState

# 모든 걸 messages에
class State(MessagesState):
    pass  # 별도 필드 없음

# 검색 결과도 그냥 messages에 욱여넣음
# state["messages"].append(
#     f"검색결과: {results}"  # 별도 필드 대신 텍스트로
# )

In [3]:
from typing import Annotated
from operator import add
from langgraph.graph import MessagesState

# 목적별 필드 + 리듀서
class State(MessagesState):
    sources: Annotated[list, add]

## Part 3. 노드에서 State 읽고 쓰기

In [4]:
# State 읽기 — 검색 노드
def search_node(state):
    q = state["messages"][-1]
    results = web_search(q.content)   # 실제 검색 함수
    return {
        "search_results": results     # 바꿀 필드만 반환
    }

# State 갱신 — 답변 노드
def answer_node(state):
    ctx = state["search_results"]
    msg = llm.invoke(ctx)
    return {
        "messages": [msg]             # add_messages 리듀서가 자동 누적
    }

## Part 4. 조건부 엣지를 활용한 분기 로직

In [5]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END, add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

# 라우터 노드: 라벨 문자열만 반환
def router(state: State) -> str:
    q = state["messages"][-1].content
    # LLM으로 질문 유형 분류
    if "계산" in q:
        return "calc"
    if "오늘" in q:
        return "realtime"
    return "knowledge"

# 각 경로의 처리 노드
def calc_node(state: State):
    return {"messages": [f"[계산] {state['messages'][-1].content}"]}

def search_node(state: State):
    return {"messages": [f"[실시간 검색] {state['messages'][-1].content}"]}

def rag_node(state: State):
    return {"messages": [f"[지식 검색(RAG)] {state['messages'][-1].content}"]}

graph2 = StateGraph(State)
graph2.add_node("router", lambda state: state)  # 라우터는 상태를 그대로 전달
graph2.add_node("calc_node", calc_node)
graph2.add_node("search_node", search_node)
graph2.add_node("rag_node", rag_node)

graph2.add_edge(START, "router")

# 반환 라벨에 따라 분기
graph2.add_conditional_edges(
    "router", router,
    {"calc": "calc_node",
     "realtime": "search_node",
     "knowledge": "rag_node"}
)
graph2.add_edge("calc_node", END)
graph2.add_edge("search_node", END)
graph2.add_edge("rag_node", END)

router_app2 = graph2.compile()

In [6]:
from langchain_core.messages import HumanMessage

questions = [
    "계산 좀 해줘, 15 + 27은?",
    "오늘 환율 얼마야?",
    "LangGraph가 뭔가요?",
]

for q in questions:
    result = router_app2.invoke({"messages": [HumanMessage(content=q)]})
    route = router({"messages": [HumanMessage(content=q)]})
    answer_msg = result["messages"][-1]
    print(f"Q  : {q}")
    print(f"경로: {route}")
    print(f"A  : {answer_msg.content if hasattr(answer_msg, 'content') else answer_msg}")
    print()

Q  : 계산 좀 해줘, 15 + 27은?
경로: calc
A  : [계산] 계산 좀 해줘, 15 + 27은?

Q  : 오늘 환율 얼마야?
경로: realtime
A  : [실시간 검색] 오늘 환율 얼마야?

Q  : LangGraph가 뭔가요?
경로: knowledge
A  : [지식 검색(RAG)] LangGraph가 뭔가요?



## Part 5. 라우터 노드로 질문 분류하기

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState

def classify(q: str) -> str:
    """질문 유형을 분류해 라우팅 키를 반환한다."""
    if any(kw in q for kw in ["날씨", "뉴스", "실시간", "지금"]):
        return "실시간"
    elif any(kw in q for kw in ["더하기", "빼기", "곱하기", "나누기", "+", "-", "*", "/"]):
        return "계산"
    else:
        return "지식"

def router(state):
    q = state["messages"][-1].content
    return classify(q)  # "지식" | "실시간" | "계산"

# 각 경로의 처리 노드
def answer(state):
    return {"messages": [f"[지식 답변] {state['messages'][-1].content}"]}

def search(state):
    return {"messages": [f"[실시간 검색] {state['messages'][-1].content}"]}

def calc(state):
    return {"messages": [f"[계산 처리] {state['messages'][-1].content}"]}

graph = StateGraph(MessagesState)
graph.add_node("router", lambda state: state)
graph.add_node("answer", answer)
graph.add_node("search", search)
graph.add_node("calc", calc)

graph.add_edge(START, "router")
graph.add_conditional_edges(
    "router",
    router,
    {"지식": "answer",
     "실시간": "search",
     "계산": "calc"}
)
graph.add_edge("answer", END)
graph.add_edge("search", END)
graph.add_edge("calc", END)

router_app = graph.compile()

In [8]:
from langchain_core.messages import HumanMessage

questions = [
    "파이썬이란 무엇인가요?",
    "오늘 서울 날씨 알려줘",
    "3 더하기 7은?",
]

for q in questions:
    result = router_app.invoke({"messages": [HumanMessage(content=q)]})
    route = classify(q)
    answer_msg = result["messages"][-1]
    print(f"Q : {q}")
    print(f"경로: {route}")
    print(f"A : {answer_msg.content if hasattr(answer_msg, 'content') else answer_msg}")
    print()

Q : 파이썬이란 무엇인가요?
경로: 지식
A : [지식 답변] 파이썬이란 무엇인가요?

Q : 오늘 서울 날씨 알려줘
경로: 실시간
A : [실시간 검색] 오늘 서울 날씨 알려줘

Q : 3 더하기 7은?
경로: 계산
A : [계산 처리] 3 더하기 7은?



## Part 6. 전체 그래프 조립하기

In [9]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 분기 함수: 반환값이 곧 다음 노드 이름
def route(state: AgentState) -> str:
    q = state["messages"][-1].content
    if "계산" in q:
        return "calc"
    if "오늘" in q or "실시간" in q:
        return "search"
    return "answer"

# 라우터 노드
def router(state: AgentState):
    return state

# 처리 노드
def search_node(state: AgentState):
    return {"messages": [f"[실시간 검색] {state['messages'][-1].content}"]}

def calc_node(state: AgentState):
    return {"messages": [f"[계산] {state['messages'][-1].content}"]}

def answer_node(state: AgentState):
    return {"messages": [f"[지식 답변] {state['messages'][-1].content}"]}

# 그래프 조립
builder = StateGraph(AgentState)

builder.add_node("router", router)
builder.add_node("search", search_node)
builder.add_node("calc", calc_node)
builder.add_node("answer", answer_node)

builder.set_entry_point("router")
builder.add_conditional_edges("router", route)

graph = builder.compile()

## Part 7. START와 END로 흐름 닫기

In [10]:
from langgraph.graph import StateGraph, START, END

# set_entry_point 대신 START 상수로 진입점 명시
graph2 = StateGraph(AgentState)
graph2.add_node("router", router)
graph2.add_node("answer", answer_node)

# 노드 간 기본 엣지 연결
graph2.add_edge("router", "answer")

# 시작과 끝을 상수로 연결
graph2.add_edge(START, "router")
graph2.add_edge("answer", END)

app2 = graph2.compile()

In [11]:
from langchain_core.messages import HumanMessage

result = app2.invoke({"messages": [HumanMessage(content="LangGraph의 START와 END란?")]})
print(result["messages"][-1].content)

[지식 답변] LangGraph의 START와 END란?


## Part 8. RAG + Function Calling 동시 등록

In [12]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

# RAG 도구: 문서에서 지식 검색 (더미 retriever)
@tool
def search_docs(query: str) -> str:
    """문서에서 관련 내용을 검색한다."""
    return f"'{query}'에 관한 문서 검색 결과: LangGraph는 상태 기반 멀티에이전트 프레임워크입니다."

# Function Calling 도구: 일반 계산 함수
@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱한다."""
    return a * b

# RAG 도구 + 일반 도구 함께 등록
tools = [search_docs, multiply]
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools = llm.bind_tools(tools)

print("등록된 도구:", [t.name for t in tools])

d:\GitHub\1_lecture-materials\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


등록된 도구: ['search_docs', 'multiply']


## Part 9. 여러 도구를 ToolNode로 묶기

In [13]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def llm_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    # Gemini 2.5 Flash는 thinking 모드로 content가 list로 올 수 있음
    # 대화 이력으로 재전달 시 오류가 생기므로 저장 전에 문자열로 정규화
    if isinstance(response.content, list):
        text = " ".join(
            block.get("text", "")
            for block in response.content
            if isinstance(block, dict) and block.get("type") == "text"
        )
        response = AIMessage(content=text, tool_calls=response.tool_calls)
    return {"messages": [response]}

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)

# 여러 도구를 한 노드로 묶음
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "llm")
# LLM이 도구를 부르면 tools로, 아니면 END
graph.add_conditional_edges("llm", tools_condition)
# 도구 실행 후 다시 LLM으로
graph.add_edge("tools", "llm")

app = graph.compile()
print("ToolNode 그래프 컴파일 완료")

ToolNode 그래프 컴파일 완료


In [14]:
import time
from langchain_core.messages import HumanMessage

def extract_text(content) -> str:
    """str 또는 콘텐츠 블록 리스트에서 텍스트를 추출한다."""
    if isinstance(content, str):
        return content
    return " ".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

questions = ["LangGraph가 뭐야?", "7 곱하기 8은?"]

for i, q in enumerate(questions):
    result = app.invoke({"messages": [HumanMessage(content=q)]})
    print(f"Q: {q}")
    print(f"A: {extract_text(result['messages'][-1].content)}")
    print()
    if i < len(questions) - 1:
        time.sleep(15)  # Gemini 무료 티어 분당 호출 제한

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 8.092801494s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

## Part 10. 대화를 기억하려면 무엇이 필요할까

In [15]:
import time
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver

class MemState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 도구 없이 순수 대화만 담당 — llm_with_tools 대신 llm 사용
# (Gemini 함수 호출 API는 멀티턴 이력 재전달 시 파싱 오류 발생)
def chat_node(state: MemState):
    response = llm.invoke(state["messages"])
    content = response.content
    if isinstance(content, list):
        content = extract_text(content)
    return {"messages": [AIMessage(content=content)]}

mem_graph = StateGraph(MemState)
mem_graph.add_node("chat", chat_node)
mem_graph.add_edge(START, "chat")
mem_graph.add_edge("chat", END)

# MemorySaver : 인메모리 체크포인터 
memory = MemorySaver()
app_with_memory = mem_graph.compile(checkpointer=memory)

# thread_id로 같은 대화를 식별
config = {"configurable": {"thread_id": "user-1"}}

# 1턴: 이름 저장
result1 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="내 이름은 민수야")]},
    config
)
print("1턴:", extract_text(result1["messages"][-1].content))

time.sleep(15)

# 2턴: 이전 State를 불러와 이어감
result2 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="내 이름 기억해?")]},
    config
)
print("2턴:", extract_text(result2["messages"][-1].content))
# → "민수"라고 답한다

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 13.766728678s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## Part 11. StateGraph로 전체 조립하기

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 분기 함수: "rag"(지식) or "llm"(도구 필요)
def route(state: AgentState) -> str:
    q = state["messages"][-1].content
    if any(kw in q for kw in ["더하기", "빼기", "곱하기", "나누기", "계산", "*", "+"]):
        return "llm"
    if any(kw in q for kw in ["날씨", "뉴스", "실시간", "지금", "오늘"]):
        return "llm"
    return "rag"

# 라우터 노드: 분기만 담당, 상태는 그대로 전달
def router(state: AgentState):
    return state

# RAG 노드: 문서 검색 후 LLM으로 답변 생성
def rag_node(state: AgentState):
    q = state["messages"][-1].content
    docs = search_docs.invoke(q)          # RAG 검색
    prompt = f"다음 문서를 참고해 답변해줘:\n{docs}\n\n질문: {q}"
    response = llm.invoke(prompt)
    if isinstance(response.content, list):
        response = AIMessage(content=extract_text(response.content))
    return {"messages": [response]}

# LLM 노드: 도구를 활용해 답변 생성
def llm_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    if isinstance(response.content, list):
        response = AIMessage(
            content=extract_text(response.content),
            tool_calls=response.tool_calls
        )
    return {"messages": [response]}

# 그래프 조립
memory = MemorySaver()

g = StateGraph(AgentState)
g.add_node("router", router)
g.add_node("rag", rag_node)
g.add_node("tools", ToolNode(tools))
g.add_node("llm", llm_node)

# 라우터가 정한 라벨로 분기
g.add_conditional_edges("router", route, {"rag": "rag", "llm": "llm"})
g.set_entry_point("router")

# llm → tools(도구 호출 시) or END
g.add_conditional_edges("llm", tools_condition)
g.add_edge("tools", "llm")   # 도구 실행 후 LLM으로 복귀
g.add_edge("rag", END)

# 기억까지 더해 완성
app = g.compile(checkpointer=memory)
print("전체 그래프 조립 완료 (RAG + Tools + Memory)")

전체 그래프 조립 완료 (RAG + Tools + Memory)


In [16]:
import time
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "user-1"}}

questions = [
    ("LangGraph가 뭐야?",      "rag"),   # 지식 → RAG 경로
    ("7 곱하기 8은?",           "llm"),  # 계산 → LLM+Tools 경로
]

for i, (q, expected_route) in enumerate(questions):
    result = app.invoke({"messages": [HumanMessage(content=q)]}, config)
    answer = extract_text(result["messages"][-1].content)
    print(f"Q      : {q}")
    print(f"예상경로: {expected_route}")
    print(f"A      : {answer}")
    print()
    if i < len(questions) - 1:
        time.sleep(15)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 22.220505338s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT